# 02 - Bronze Bookings Clean Stage Refresh

Sanitized portfolio notebook for flattening raw booking JSON from the Bronze raw table into a cleaner stage table.

Before running this notebook in a real Databricks workspace, replace placeholder paths and confirm that the Unity Catalog schemas exist.
All outputs have been cleared before publishing.


In [ ]:
from delta.tables import DeltaTable
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    upper,
    to_timestamp,
    to_date,
    current_timestamp,
    row_number,
    coalesce
)
from pyspark.sql.window import Window

PROCESS_NAME = "bronze_clean_stage"
WATERMARK_TABLE = "corptravel_databricks.control.pipeline_watermark"
SOURCE_TABLE = "corptravel_databricks.bronze.bookings_raw"
TARGET_TABLE = "corptravel_databricks.bronze.bookings_clean_stage"

# =========================================================
# 1. READ LAST WATERMARK
# =========================================================
watermark_df = (
    spark.table(WATERMARK_TABLE)
    .filter(col("process_name") == PROCESS_NAME)
)

last_watermark = watermark_df.collect()[0]["last_ingest_time"]

print(f"Last watermark for {PROCESS_NAME}: {last_watermark}")

# =========================================================
# 2. READ ONLY NEW RAW ROWS
# =========================================================
new_raw_df = (
    spark.table(SOURCE_TABLE)
    .filter(col("ingest_time") > last_watermark)
)

if new_raw_df.limit(1).count() == 0:
    print("No new raw rows found. Nothing to process.")
else:
    print(f"New raw rows found after watermark: {new_raw_df.count()}")

    # =========================================================
    # 3. FIND CHANGED BOOKING IDS
    # =========================================================
    changed_booking_ids_df = (
        new_raw_df
        .select(upper(trim(col("bookingId"))).alias("booking_id"))
        .where(col("booking_id").isNotNull())
        .distinct()
    )

    print(f"Changed booking IDs: {changed_booking_ids_df.count()}")

    # =========================================================
    # 4. RE-READ FULL RAW TABLE ONLY FOR CHANGED BOOKING IDS
    # =========================================================
    bronze_df = spark.table(SOURCE_TABLE)

    impacted_raw_df = (
        bronze_df.alias("b")
        .join(
            changed_booking_ids_df.alias("c"),
            upper(trim(col("b.bookingId"))) == col("c.booking_id"),
            "inner"
        )
    )

    # =========================================================
    # 5. TRANSFORM TO CLEAN STAGE FORMAT
    # =========================================================
    clean_stage_df = (
        impacted_raw_df
        .select(
            upper(trim(col("b.bookingId"))).alias("booking_id"),

            trim(col("b.employee.employeeId")).alias("employee_id"),
            trim(col("b.employee.name")).alias("employee_name"),
            trim(col("b.employee.department")).alias("department"),
            lower(trim(col("b.employee.email"))).alias("employee_email"),

            trim(col("b.travelDetails.tripType")).alias("trip_type"),
            trim(col("b.travelDetails.origin")).alias("origin"),
            trim(col("b.travelDetails.destination")).alias("destination"),
            to_timestamp(col("b.travelDetails.departureDate")).alias("departure_ts"),
            to_timestamp(col("b.travelDetails.returnDate")).alias("return_ts"),
            trim(col("b.travelDetails.mode")).alias("travel_mode"),
            trim(col("b.travelDetails.airline")).alias("airline"),

            trim(col("b.hotelDetails.hotelName")).alias("hotel_name"),
            to_date(col("b.hotelDetails.checkIn")).alias("check_in"),
            to_date(col("b.hotelDetails.checkOut")).alias("check_out"),
            trim(col("b.hotelDetails.city")).alias("hotel_city"),

            upper(trim(col("b.costDetails.currency"))).alias("currency"),
            col("b.costDetails.flightCost").cast("double").alias("flight_cost"),
            col("b.costDetails.hotelCost").cast("double").alias("hotel_cost"),
            col("b.costDetails.totalCost").cast("double").alias("total_cost"),

            trim(col("b.approval.status")).alias("approval_status"),
            trim(col("b.approval.approvedBy")).alias("approved_by"),
            to_timestamp(col("b.approval.approvalDate")).alias("approval_ts"),

            trim(col("b.bookingStatus")).alias("booking_status"),
            to_timestamp(col("b.BookingCreatedAt")).alias("booking_created_ts"),
            lower(trim(col("b.pocEmail"))).alias("poc_email"),

            # New metadata columns for stronger incremental logic
            to_timestamp(col("b.updatedAt")).alias("updated_ts"),
            to_timestamp(col("b.rawStoredAt")).alias("raw_stored_ts"),
            col("b.rawEventId").alias("raw_event_id"),

            col("b.source_file").alias("source_file"),
            col("b.ingest_time").alias("ingest_time"),
            current_timestamp().alias("silver_processed_at")
        )
    )

    # =========================================================
    # 6. KEEP ONLY LATEST VERSION PER BOOKING_ID
    # =========================================================
    latest_window = Window.partitionBy("booking_id").orderBy(
        coalesce(col("updated_ts"), col("ingest_time")).desc(),
        col("ingest_time").desc()
    )

    latest_clean_stage_df = (
        clean_stage_df
        .withColumn("rn", row_number().over(latest_window))
        .filter(col("rn") == 1)
        .drop("rn")
    )

    # =========================================================
    # 7. CREATE TARGET TABLE IF NOT EXISTS
    # =========================================================
    if not spark.catalog.tableExists(TARGET_TABLE):
        latest_clean_stage_df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(TARGET_TABLE)
        print(f"Created target table: {TARGET_TABLE}")
    else:
        target = DeltaTable.forName(spark, TARGET_TABLE)

        target.alias("t").merge(
            latest_clean_stage_df.alias("s"),
            "t.booking_id = s.booking_id"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()

        print(f"Merged changed rows into: {TARGET_TABLE}")

    # =========================================================
    # 8. UPDATE WATERMARK
    # =========================================================
    max_ingest_time = new_raw_df.agg({"ingest_time": "max"}).collect()[0][0]

    spark.sql(f"""
        DELETE FROM {WATERMARK_TABLE}
        WHERE process_name = '{PROCESS_NAME}'
    """)

    spark.sql(f"""
        INSERT INTO {WATERMARK_TABLE}
        VALUES ('{PROCESS_NAME}', TIMESTAMP('{max_ingest_time}'))
    """)

    print(f"Updated watermark for {PROCESS_NAME} to {max_ingest_time}")